# 从零理解 PyTorch Linear

本 notebook 配合 CS336 Assignment 1 使用，目标是理解并自己完成一个**无 bias 的 Linear 层**。重点包括：

- `nn.Module` 为什么是所有模型层的基类
- `nn.Parameter` 与普通 Tensor 的区别
- `torch.empty`、截断正态初始化和原地操作
- `forward`、`__call__` 与自动求导
- 为什么输入行数和前导维度不需要提前确定
- `state_dict`、设备、数据类型和测试权重加载

建议按顺序运行所有单元，并先独立完成习题，再展开参考答案。

In [1]:
import math
import torch
from torch import nn

torch.manual_seed(42)
print('PyTorch version:', torch.__version__)

PyTorch version: 2.11.0+cpu


## 1. Linear 在计算什么？

数学中常把一个输入写成列向量，线性变换写作：

$$y = Wx$$

PyTorch 通常把一个样本放在一行，多个样本叠成 `(..., in_features)`，因此写作：

$$y = xW^T$$

权重约定为 `W.shape == (out_features, in_features)`。每一行对应一个输出神经元的权重。

```text
x:        (..., in_features)
W:        (out_features, in_features)
W.T:      (in_features, out_features)
x @ W.T:  (..., out_features)
```

`...` 可以是空、batch 维、`(batch, sequence)`，甚至更多维。Linear 只约束最后一维。

In [3]:
W = torch.tensor([[1.0, 2.0, 3.0], [-1.0, 0.0, 1.0]])  # (out=2, in=3)
x = torch.tensor([[1.0, 10.0, 100.0], [2.0, 20.0, 200.0]])  # (batch=2, in=3)
y = x @ W.T

print('x shape:', x.shape)
print('W shape:', W.shape)
print('W.T shape:', W.T.shape)
print('y shape:', y.shape)
print(y)

assert y.shape == (2, 2)

x shape: torch.Size([2, 3])
W shape: torch.Size([2, 3])
W.T shape: torch.Size([3, 2])
y shape: torch.Size([2, 2])
tensor([[321.,  99.],
        [642., 198.]])


## 2. `nn.Module`：模型组件的基础设施

自定义层通常继承 `nn.Module`。它提供的能力包括：

1. 递归注册参数和子模块。
2. `.parameters()` 给优化器提供参数。
3. `.state_dict()` 保存模型状态。
4. `.to(device/dtype)` 统一迁移参数。
5. `layer(x)` 调用 `Module.__call__`，再由它调用 `forward(x)`。

必须在构造函数中调用 `super().__init__()`，否则 Module 内部用于注册参数和模块的容器尚未建立。

不要自己重写 `__call__`。PyTorch 的 `__call__` 还负责 hooks、autocast 等机制；我们只实现 `forward`。

### 2.1 
`torch.nn` 是 PyTorch 的神经网络工具包，通常这样导入：

```python
from torch import nn
```

可以把一个 Module 暂时理解成一个“会计算、会管理自身状态的 Python 对象”：

- **计算规则**写在 `forward` 中。
- **可训练数据**是 Parameter，例如 Linear 的权重。
- **内部组件**可以是其他 Module，例如一个模型包含多个 Linear。
- **状态管理**由 Module 提供，例如保存、加载、迁移设备、切换训练模式。

Module 不一定是完整模型。一个 Linear 层是 Module，一个 Transformer Block 是 Module，整个 Transformer 也是 Module。它们像积木一样可以逐层组合。

### 2.2 从普通 Python 类走到 Module

先看一个普通 Python 类：

```python
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def calculate(self, x):
        return x * self.factor
```

`__init__` 在创建对象时执行，用于保存对象需要的数据；`self` 表示当前对象。`nn.Module` 仍然遵循这些普通 Python 规则，只是额外提供了神经网络需要的管理能力。

自定义 Module 的最小结构是：

```python
class MyModule(nn.Module):
    def __init__(self):
        super().__init__()
        # 在这里创建参数或子模块

    def forward(self, x):
        # 在这里描述输入如何变成输出
        return x
```

`super().__init__()` 会运行父类 `nn.Module` 的初始化代码，建立 `_parameters`、`_modules`、`_buffers` 等内部容器。它必须先执行，然后才能给对象赋 Parameter 或子 Module。

In [4]:
class Double(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 2 * x

double = Double()             # 创建 Module 对象
x_demo = torch.tensor([1.0, 2.0, 3.0])
y_demo = double(x_demo)       # 日常使用方式：对象(输入)

print('module:', double)
print('input:', x_demo)
print('output:', y_demo)
print('parameters:', list(double.parameters()))

assert torch.equal(y_demo, torch.tensor([2.0, 4.0, 6.0]))
assert len(list(double.parameters())) == 0  # Module 可以没有可训练参数

module: Double()
input: tensor([1., 2., 3.])
output: tensor([2., 4., 6.])
parameters: []


### 2.3 `layer(x)` 到底发生了什么？

`layer(x)` 看起来像调用函数，实际上 Python 会执行 `layer.__call__(x)`。因为 `layer` 继承了 `nn.Module`，这里使用的是 PyTorch 已实现的 `Module.__call__`。简化后的流程可以理解为：

```text
layer(x)
  -> Module.__call__(x)
      -> 执行 forward 前的 hooks
      -> layer.forward(x)
      -> 执行 forward 后的 hooks
      -> 返回结果
```

所以我们实现 `forward`，但调用时写 `layer(x)`。直接写 `layer.forward(x)` 可能绕开 hooks 和 PyTorch 的其他包装逻辑。

`forward` 本身没有神秘之处：它通常只是由 `+`、`@`、激活函数等 Tensor 运算组成。只要这些运算支持 autograd，PyTorch 就能在运行时自动建立计算图。

In [5]:
events = []

def before_forward(module, inputs):
    events.append(('before', tuple(inputs[0].shape)))

def after_forward(module, inputs, output):
    events.append(('after', tuple(output.shape)))

double = Double()
pre_handle = double.register_forward_pre_hook(before_forward)
post_handle = double.register_forward_hook(after_forward)
_ = double(torch.ones(2, 3))
pre_handle.remove()
post_handle.remove()

print(events)
assert events == [('before', (2, 3)), ('after', (2, 3))]

[('before', (2, 3)), ('after', (2, 3))]


### 2.4 Module 如何自动管理子模块？

当你把另一个 Module 赋给当前 Module 的属性时，它会被自动注册为**子模块**：

```python
self.input_layer = nn.Linear(3, 4)
```

这会形成一棵模块树。父模块的 `.parameters()` 会递归找到所有子模块的 Parameter；`.to()`、`.train()`、`.eval()` 和 `.state_dict()` 也会沿着这棵树递归工作。

大量同类子模块应放进 `nn.ModuleList` 或 `nn.Sequential`，不能随手放进普通 list 后期待 PyTorch 自动发现。

- `nn.Sequential(a, b)`：按顺序执行 `b(a(x))`。
- `nn.ModuleList([a, b])`：只负责注册，具体执行顺序由你在 `forward` 中编写。

Transformer 的多层 Block 常使用 `ModuleList`，因为 forward 往往还包含残差连接等自定义逻辑。

In [ ]:
class TinyNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(3, 4),
            nn.ReLU(),
            nn.Linear(4, 2),
        )

    def forward(self, x):
        return self.layers(x)

network = TinyNetwork()
print(network)
print('\n子模块名称：')
for name, module in network.named_modules():
    display_name = name if name else '<root>'
    print(f'{display_name}: {type(module).__name__}')

print('\n参数名称和形状：')
for name, parameter in network.named_parameters():
    print(name, tuple(parameter.shape))

parameter_count = sum(parameter.numel() for parameter in network.parameters())
print('参数总数：', parameter_count)
# (3*4 + 4) + (4*2 + 2) = 26；PyTorch 自带 Linear 默认有 bias。
assert parameter_count == 26
assert network(torch.randn(5, 3)).shape == (5, 2)

### 2.5 Module 的训练状态：`train()` 与 `eval()`

每个 Module 都有布尔属性 `training`。

- `model.train()` 将模型和所有子模块切换为训练模式。
- `model.eval()` 将模型和所有子模块切换为评估模式。

它们**不会开始训练，也不会关闭梯度**，只是改变 Dropout、BatchNorm 等少数层的行为。Linear 在两种模式下计算相同。

是否记录梯度由 `torch.no_grad()` 或 `torch.inference_mode()` 控制。因此推理时常同时写：

```python
model.eval()
with torch.no_grad():
    prediction = model(x)
```

In [ ]:
dropout = nn.Dropout(p=0.5)
ones = torch.ones(12)

dropout.train()
train_output = dropout(ones)
dropout.eval()
eval_output = dropout(ones)

print('training flag:', dropout.training)
print('训练模式输出:', train_output)
print('评估模式输出:', eval_output)
assert torch.equal(eval_output, ones)

### 2.6 初学者自测

先不看答案，判断下面说法：

1. 继承 `nn.Module` 后，Python 会自动替你写 `forward`。
2. 一个 Module 必须至少有一个 Parameter。
3. 调用 `model.eval()` 会关闭梯度计算。
4. 父模块的 `.parameters()` 能找到已注册子模块中的参数。
5. `model(x)` 最终会调用你实现的 `forward(x)`。
6. 把多个层放入普通 Python list，父模块一定能自动注册它们。

<details>
<summary>展开答案</summary>

1. 错。你必须实现计算规则。
2. 错。上面的 `Double` 没有参数，但仍是合法 Module。
3. 错。`eval()` 只切换层的行为；使用 `no_grad()` 才是不记录梯度。
4. 对。Module 会递归遍历注册的模块树。
5. 对，但中间经过 `Module.__call__`。
6. 错。应使用 `ModuleList`、`Sequential`，或把子模块直接赋给属性。

</details>

## 3. `nn.Parameter` 与普通 Tensor

`nn.Parameter` 是 Tensor 的子类，默认 `requires_grad=True`。把 Parameter 赋给 Module 的属性时，它会被自动注册。普通 Tensor 属性不会被视为模型参数。

注册意味着它会出现在：

- `named_parameters()`
- `parameters()`
- `state_dict()`
- 优化器的更新对象中

注意：`requires_grad=True` 的普通 Tensor 仍不会自动注册为 Module 参数；注册身份来自 `nn.Parameter`。

In [6]:
class ParameterDemo(nn.Module):
    def __init__(self):
        super().__init__()
        self.trainable = nn.Parameter(torch.tensor([1.0, 2.0]))
        self.not_registered = torch.tensor([3.0, 4.0], requires_grad=True)
        # buffer 是需要保存/迁移但不由优化器训练的状态。
        self.register_buffer('running_value', torch.tensor([5.0]))

demo = ParameterDemo()
print('named_parameters:', list(demo.named_parameters()))
print('state_dict keys:', list(demo.state_dict().keys()))

assert 'trainable' in dict(demo.named_parameters())
assert 'not_registered' not in dict(demo.named_parameters())
assert 'running_value' in demo.state_dict()

named_parameters: [('trainable', Parameter containing:
tensor([1., 2.], requires_grad=True))]
state_dict keys: ['trainable', 'running_value']


## 4. 创建和初始化权重

`torch.empty(shape)` 只分配内存，不保证其中的值有意义，所以必须紧接初始化。CS336 的 Linear 使用：

$$\sigma = \sqrt{\frac{2}{d_{in}+d_{out}}}$$

然后从均值 0、标准差 $\sigma$ 的正态分布采样，并截断到 $[-3\sigma, 3\sigma]$。

`nn.init.trunc_normal_` 名字末尾的 `_` 表示**原地操作**：它修改传入张量，不创建新的权重张量。原地初始化可以保留 Parameter 的身份。

In [ ]:
d_in, d_out = 64, 128
sigma = math.sqrt(2 / (d_in + d_out))
weight = nn.Parameter(torch.empty(d_out, d_in))
nn.init.trunc_normal_(weight, mean=0.0, std=sigma, a=-3 * sigma, b=3 * sigma)

print('sigma:', sigma)
print('observed min/max:', weight.min().item(), weight.max().item())
assert weight.shape == (d_out, d_in)
assert weight.min() >= -3 * sigma
assert weight.max() <= 3 * sigma

## 5. 一个完整的教学版 Linear

下面的类展示全部机制。它没有 bias，且没有调用 `nn.Linear`。`device` 和 `dtype` 被传给 `torch.empty`，因此参数从创建时就在正确设备上、使用正确类型。

In [ ]:
class TeachingLinear(nn.Module):
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(
            torch.empty((out_features, in_features), device=device, dtype=dtype)
        )
        sigma = math.sqrt(2 / (in_features + out_features))
        nn.init.trunc_normal_(
            self.weight, mean=0.0, std=sigma, a=-3 * sigma, b=3 * sigma
        )

    def forward(self, x):
        return x @ self.weight.T

layer = TeachingLinear(3, 5)
print(layer)
print('weight shape:', layer.weight.shape)
print('parameters:', [(name, tuple(p.shape)) for name, p in layer.named_parameters()])

## 6. 行数未知和任意前导维度

Linear 不关心输入有多少行，只要求 `x.shape[-1] == in_features`。矩阵乘法 `@`（即 `torch.matmul`）将最后两个维度用于矩阵乘法，并把其余维度当作批次维度。

因此同一个 `Linear(3, 5)` 可以接受：

- `(3,) -> (5,)`：单个向量
- `(8, 3) -> (8, 5)`：8 个样本
- `(4, 10, 3) -> (4, 10, 5)`：4 个序列，每个序列 10 个 token

这里没有循环；矩阵乘法内核会批量完成计算。

In [ ]:
layer = TeachingLinear(3, 5)
for shape in [(3,), (8, 3), (4, 10, 3), (2, 4, 10, 3)]:
    x = torch.randn(shape)
    y = layer(x)  # 等价于 layer.forward(x)，但日常应调用 layer(x)
    print(f'{shape} -> {tuple(y.shape)}')
    assert y.shape == x.shape[:-1] + (5,)

## 7. 自动求导：Parameter 如何得到梯度

前向计算由可微 Tensor 运算组成时，PyTorch 自动建立计算图。调用 `loss.backward()` 后：

- `x.grad` 保存损失对输入的梯度（前提是输入要求梯度）。
- `weight.grad` 保存损失对权重的梯度。
- 梯度默认累加，因此训练循环中通常在下一次反向传播前调用 `optimizer.zero_grad()`。

In [ ]:
layer = TeachingLinear(3, 2)
x = torch.randn(4, 3, requires_grad=True)
target = torch.zeros(4, 2)

prediction = layer(x)
loss = ((prediction - target) ** 2).mean()
loss.backward()

print('loss:', loss.item())
print('x.grad shape:', x.grad.shape)
print('weight.grad shape:', layer.weight.grad.shape)
assert x.grad.shape == x.shape
assert layer.weight.grad.shape == layer.weight.shape

## 8. `state_dict`、`copy_` 和 `torch.no_grad()`

测试适配器会提供指定权重。应该把数值复制进已有 Parameter：

```python
with torch.no_grad():
    layer.weight.copy_(weights)
```

为什么这样做？

- `copy_` 修改原 Parameter 的数值，保留注册身份。
- `torch.no_grad()` 表示这是模型配置，不应进入计算图。
- 不应写 `layer.weight = weights`，因为右侧通常只是 Tensor，Module 会拒绝或丢失参数语义。
- 也可使用 `load_state_dict({'weight': weights})`，但适配器只有一个参数时 `copy_` 更直接。

In [ ]:
layer = TeachingLinear(3, 2)
provided_weight = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
parameter_id_before = id(layer.weight)

with torch.no_grad():
    layer.weight.copy_(provided_weight)

print('state_dict:', layer.state_dict())
print('output:', layer(torch.tensor([[10.0, 20.0, 30.0]])))
assert id(layer.weight) == parameter_id_before
assert isinstance(layer.weight, nn.Parameter)

## 9. `device` 和 `dtype`

矩阵乘法通常要求输入和权重位于同一设备，并使用兼容的数据类型。构造参数支持 `device`、`dtype`，可以避免先在 CPU 创建再迁移。

```python
layer = TeachingLinear(64, 128, device='cuda', dtype=torch.float16)
```

也可以事后迁移：`layer.to(device='cuda', dtype=torch.float16)`。`.to()` 会递归处理已注册的 Parameter 和 buffer；普通 Tensor 属性不会自动迁移，这也是正确注册状态的重要性之一。

In [ ]:
layer64 = TeachingLinear(3, 2, dtype=torch.float64)
x64 = torch.randn(4, 3, dtype=torch.float64)
y64 = layer64(x64)
print('weight dtype:', layer64.weight.dtype)
print('output dtype:', y64.dtype)
assert y64.dtype == torch.float64

## 10. 常见错误

### 错误 1：权重形状写反

作业要求 `(out_features, in_features)`，所以 forward 使用 `x @ weight.T`。如果权重创建成 `(in, out)`，即使乘法能跑，也不符合测试提供权重的格式。

### 错误 2：忘记 `super().__init__()`

Parameter 无法被正常注册。

### 错误 3：只用普通 Tensor 保存权重

优化器和 `state_dict` 找不到它。

### 错误 4：`torch.empty` 后不初始化

其中可能包含任意内存值，输出不可控。

### 错误 5：手动遍历 batch 或 sequence

不需要循环。`@` 已支持任意前导维度，而且更快。

### 错误 6：用 `.data` 修改参数

`.data` 容易绕过 autograd 的安全检查。配置权重时优先使用 `with torch.no_grad(): parameter.copy_(...)`。

## 11. 习题

### 习题 1：形状推导

给定 `x.shape == (8, 128, 64)`，`weight.shape == (256, 64)`：

1. `weight.T` 是什么形状？
2. `x @ weight.T` 是什么形状？
3. 哪些维度不被 Linear 改变？

### 习题 2：Parameter 实验

创建一个 Module，同时保存普通 Tensor、`requires_grad=True` 的 Tensor、Parameter 和 buffer。预测它们是否出现在 `named_parameters()` 与 `state_dict()`，然后运行代码验证。

### 习题 3：实现 `StudentLinear`

补全下一个代码单元。要求不使用 `nn.Linear`，权重无 bias，并通过所有断言。

### 习题 4：梯度检查

令损失为 `layer(x).sum()`，运行两次 `backward()` 且中间不清零。第二次之后 `weight.grad` 与第一次相比有什么关系？解释为什么。

### 习题 5：迁移到作业

将 `StudentLinear` 中你理解后的实现迁移到 `cs336_basics/model.py` 的两个 TODO，然后在 Linux 运行：

```bash
uv run pytest tests/test_model.py::test_linear -q
```

In [ ]:
class StudentLinear(nn.Module):
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__()
        # TODO 1: 创建形状为 (out_features, in_features) 的 Parameter。
        # self.weight = ...

        # TODO 2: 计算 sigma，并执行截断正态初始化。
        # sigma = ...
        # ...

    def forward(self, x):
        # TODO 3: 返回形状 (..., out_features) 的结果。
        raise NotImplementedError

# 完成后取消下面代码的注释：
# student_layer = StudentLinear(3, 7)
# student_x = torch.randn(2, 5, 3, requires_grad=True)
# student_y = student_layer(student_x)
# assert student_layer.weight.shape == (7, 3)
# assert student_y.shape == (2, 5, 7)
# assert 'weight' in student_layer.state_dict()
# student_y.sum().backward()
# assert student_layer.weight.grad is not None
# print('全部断言通过！')

<details>
<summary>展开查看习题提示与参考答案</summary>

### 习题 1

`weight.T` 为 `(64, 256)`，输出为 `(8, 128, 256)`。`8` 和 `128` 是前导维度，保持不变。

### 习题 3 提示

需要用到：`nn.Parameter`、`torch.empty`、`math.sqrt`、`nn.init.trunc_normal_`、`.T` 和 `@`。

核心前向表达式只有一行：输入乘以权重的转置。请先自己写，再和上面的 `TeachingLinear` 对照。

### 习题 4

第二次梯度通常是第一次的两倍，因为 `.backward()` 把新梯度累加到已有 `.grad`，不会自动覆盖。训练循环应在适当位置调用 `optimizer.zero_grad()` 或把梯度设为 `None`。

</details>

## 12. 完成检查表

完成作业的 Linear 前，确认你能回答：

- 为什么权重是 `(out, in)`，前向却是 `x @ weight.T`？
- 为什么 batch 大小和 sequence 长度不需要写进构造函数？
- Parameter 比普通 Tensor 多了什么 Module 语义？
- 为什么 `torch.empty` 后必须初始化？
- 为什么载入测试权重要用 `no_grad + copy_`？
- 为什么应调用 `layer(x)` 而不是直接调用 `forward(x)`？

如果这些问题都能独立解释，你已经掌握了实现 Linear 所需的 PyTorch 核心机制。